# Lakekeeper AuthN to OpenFGA AuthZ POC



Run this notebook against the base `docker-compose.yaml` stack first. It proves Keycloak authentication independently of authorization, loads deterministic data, and records the in-place OpenFGA cutover. Run the host-side commands only when the notebook tells you to pause.

## 1. Configure POC Environment



The defaults target this example's Docker network. Override them with environment variables when applying the procedure to another deployment.

In [ ]:
%pip install -q pyiceberg pandas pyarrow trino requests pyjwt



import os

import time

import jwt

import requests

import pyarrow as pa

from IPython.display import JSON, display

from pyiceberg.catalog.rest import RestCatalog

from trino.dbapi import connect



CATALOG_URL = os.getenv("CATALOG_URL", "http://lakekeeper:8181/catalog")

MANAGEMENT_URL = os.getenv("MANAGEMENT_URL", "http://lakekeeper:8181/management")

TRINO_HOST = os.getenv("TRINO_HOST", "trino")

TRINO_PORT = int(os.getenv("TRINO_PORT", "8080"))

KEYCLOAK_ISSUER = os.getenv("KEYCLOAK_ISSUER", "http://keycloak:8080/realms/iceberg")

TOKEN_URL = f"{KEYCLOAK_ISSUER}/protocol/openid-connect/token"

WAREHOUSE = os.getenv("WAREHOUSE", "demo")



CLIENTS = {

    "technical": (os.getenv("TECHNICAL_CLIENT_ID", "spark"), os.getenv("TECHNICAL_CLIENT_SECRET", "2OR3eRvYfSZzzZ16MlPd95jhLnOaLM52")),

    "trino": (os.getenv("TRINO_CLIENT_ID", "trino"), os.getenv("TRINO_CLIENT_SECRET", "AK48QgaKsqdEpP9PomRJw7l2T7qWGHdZ")),

}

USERS = {

    "peter": (os.getenv("PETER_USERNAME", "peter"), os.getenv("PETER_PASSWORD", "iceberg")),

    "anna": (os.getenv("ANNA_USERNAME", "anna"), os.getenv("ANNA_PASSWORD", "iceberg")),

}



for url in (f"{KEYCLOAK_ISSUER}/.well-known/openid-configuration", f"{MANAGEMENT_URL}/v1/info"):

    response = requests.get(url, timeout=10)

    assert response.status_code in (200, 401), (url, response.status_code, response.text)

print("Keycloak and Lakekeeper are reachable")

## 2. Obtain Keycloak Access Tokens



The technical and Trino identities use client credentials. Peter and Anna use password grant only to keep this local POC self-contained; use your production interactive flow for human users. Token values are never displayed.

In [ ]:
def client_token(client_id, client_secret):

    response = requests.post(TOKEN_URL, data={

        "grant_type": "client_credentials", "client_id": client_id,

        "client_secret": client_secret, "scope": "lakekeeper",

    }, timeout=10)

    response.raise_for_status()

    return response.json()["access_token"]



def user_token(username, password):

    client_id, client_secret = CLIENTS["trino"]

    response = requests.post(TOKEN_URL, data={

        "grant_type": "password", "client_id": client_id,

        "client_secret": client_secret, "username": username,

        "password": password, "scope": "openid lakekeeper",

    }, timeout=10)

    response.raise_for_status()

    return response.json()["access_token"]



tokens = {name: client_token(*credentials) for name, credentials in CLIENTS.items()}

tokens.update({name: user_token(*credentials) for name, credentials in USERS.items()})



claims = {name: jwt.decode(token, options={"verify_signature": False}) for name, token in tokens.items()}

for name, token_claims in claims.items():

    assert token_claims["iss"] == KEYCLOAK_ISSUER

    audience = token_claims["aud"] if isinstance(token_claims["aud"], list) else [token_claims["aud"]]

    assert "lakekeeper" in audience

    assert token_claims["sub"]

    assert token_claims["exp"] > time.time()

display(JSON({name: {key: value for key, value in value.items() if key in ("iss", "aud", "sub", "preferred_username", "exp")} for name, value in claims.items()}))

## 3. Bootstrap Lakekeeper



The technical client becomes the initial administrator. This cell is idempotent for repeated baseline runs.

In [ ]:
def auth(token):

    return {"Authorization": f"Bearer {token}"}



info = requests.get(f"{MANAGEMENT_URL}/v1/info", headers=auth(tokens["technical"]), timeout=10)

info.raise_for_status()

if not info.json()["bootstrapped"]:

    response = requests.post(

        f"{MANAGEMENT_URL}/v1/bootstrap",

        headers=auth(tokens["technical"]),

        json={"accept-terms-of-use": True},

        timeout=30,

    )

    response.raise_for_status()



info = requests.get(f"{MANAGEMENT_URL}/v1/info", headers=auth(tokens["technical"]), timeout=10)

info.raise_for_status()

assert info.json()["bootstrapped"] is True

display(JSON(info.json()))

## 4. Create Warehouse and Sample Iceberg Data



Create the `demo` warehouse if needed, then use the technical client to create deterministic sample data through PyIceberg.

In [ ]:
response = requests.get(f"{MANAGEMENT_URL}/v1/warehouse", headers=auth(tokens["technical"]), timeout=10)

response.raise_for_status()

warehouses = response.json()["warehouses"]

warehouse = next((item for item in warehouses if item["warehouse-name"] == WAREHOUSE), None)



if warehouse is None:

    response = requests.post(f"{MANAGEMENT_URL}/v1/warehouse", headers=auth(tokens["technical"]), json={

        "warehouse-name": WAREHOUSE,

        "storage-profile": {

            "type": "s3", "bucket": "examples", "key-prefix": "initial-warehouse",

            "endpoint": "http://silo:9000", "sts-endpoint": "http://silo:9000",

            "region": "local-01", "path-style-access": True,

            "flavor": "s3-compat", "sts-enabled": True,

        },

        "storage-credential": {

            "type": "s3", "credential-type": "access-key",

            "access-key-id": "silo-root-user", "secret-access-key": "silo-root-password",

        },

    }, timeout=30)

    response.raise_for_status()

    warehouse = response.json()



warehouse_id = warehouse["warehouse-id"]

print("Warehouse:", WAREHOUSE, warehouse_id)



technical_id, technical_secret = CLIENTS["technical"]

catalog = RestCatalog(

    name=WAREHOUSE, warehouse=WAREHOUSE, uri=CATALOG_URL,

    credential=f"{technical_id}:{technical_secret}",

    **{"oauth2-server-uri": TOKEN_URL, "scope": "lakekeeper"},

)

namespace = ("authn_poc",)

table_id = ("authn_poc", "sample_events")

if namespace not in catalog.list_namespaces():

    catalog.create_namespace(namespace)

if table_id in catalog.list_tables(namespace):

    catalog.drop_table(table_id)

sample = pa.table({

    "event_id": pa.array([1, 2, 3], type=pa.int64()),

    "source": ["pyiceberg", "pyiceberg", "trino-ready"],

    "amount": pa.array([10.5, 20.0, 31.5], type=pa.float64()),

})

table = catalog.create_table(table_id, schema=sample.schema)

table.append(sample)

print("Created authn_poc.sample_events")

## 5. Query Data with PyIceberg

In [ ]:
pyiceberg_rows = catalog.load_table(table_id).scan().to_arrow().to_pylist()

expected_rows = [

    {"event_id": 1, "source": "pyiceberg", "amount": 10.5},

    {"event_id": 2, "source": "pyiceberg", "amount": 20.0},

    {"event_id": 3, "source": "trino-ready", "amount": 31.5},

]

assert pyiceberg_rows == expected_rows

pyiceberg_rows

## 6. Query Data with Trino



Trino authenticates its Iceberg REST catalog independently with its own Keycloak client. This example deliberately uses one query-engine identity rather than forwarding end-user identity.

In [ ]:
trino_client_id, trino_client_secret = CLIENTS["trino"]

connection = connect(host=TRINO_HOST, port=TRINO_PORT, user="poc")

cursor = connection.cursor()

cursor.execute("DROP CATALOG IF EXISTS lakekeeper")

cursor.execute(f'''CREATE CATALOG lakekeeper USING iceberg WITH (

    "iceberg.catalog.type" = 'rest',

    "iceberg.rest-catalog.uri" = '{CATALOG_URL}',

    "iceberg.rest-catalog.warehouse" = '{WAREHOUSE}',

    "iceberg.rest-catalog.security" = 'OAUTH2',

    "iceberg.rest-catalog.oauth2.credential" = '{trino_client_id}:{trino_client_secret}',

    "iceberg.rest-catalog.oauth2.scope" = 'lakekeeper offline_access',

    "iceberg.rest-catalog.oauth2.server-uri" = '{TOKEN_URL}',

    "iceberg.rest-catalog.vended-credentials-enabled" = 'true',

    "s3.region" = 'dummy',

    "s3.path-style-access" = 'true',

    "s3.endpoint" = 'http://silo:9000',

    "fs.native-s3.enabled" = 'true'

)''')



connection = connect(host=TRINO_HOST, port=TRINO_PORT, user="poc", catalog="lakekeeper")

cursor = connection.cursor()

namespaces = cursor.execute("SHOW SCHEMAS").fetchall()

rows = cursor.execute("SELECT event_id, source, amount FROM authn_poc.sample_events ORDER BY event_id").fetchall()

totals = cursor.execute("SELECT source, sum(amount) FROM authn_poc.sample_events GROUP BY source ORDER BY source").fetchall()

filtered = cursor.execute("SELECT event_id FROM authn_poc.sample_events WHERE amount >= 20 ORDER BY event_id").fetchall()

assert rows == [(1, "pyiceberg", 10.5), (2, "pyiceberg", 20.0), (3, "trino-ready", 31.5)]

assert filtered == [(2,), (3,)]

{"schemas": namespaces, "rows": rows, "totals": totals, "filtered": filtered}

## 7. Verify Authentication Enforcement



These checks isolate authentication from authorization. With the `allowall` backend, every valid identity can reach the catalog; absent, malformed, and expired credentials must still fail. The successful Trino queries above verify Trino's Keycloak client flow.

In [ ]:
config_url = f"{CATALOG_URL}/v1/config?warehouse={WAREHOUSE}"

authn_results = {}

for name, headers in {

    "no token": {},

    "malformed token": auth("not-a-jwt"),

    "expired token": auth(jwt.encode({"iss": KEYCLOAK_ISSUER, "aud": "lakekeeper", "sub": "expired", "exp": 1}, "not-the-keycloak-key", algorithm="HS256")),

    "technical": auth(tokens["technical"]),

    "trino": auth(tokens["trino"]),

    "peter": auth(tokens["peter"]),

    "anna": auth(tokens["anna"]),

}.items():

    response = requests.get(config_url, headers=headers, timeout=10)

    authn_results[name] = response.status_code



assert authn_results["no token"] == 401

assert authn_results["malformed token"] == 401

assert authn_results["expired token"] == 401

for name in ("technical", "trino", "peter", "anna"):

    assert authn_results[name] == 200, (name, authn_results[name])



def catalog_for_token(name):

    return RestCatalog(name=f"demo-{name}", warehouse=WAREHOUSE, uri=CATALOG_URL, token=tokens[name])



for name in ("peter", "anna"):

    assert len(catalog_for_token(name).load_table(table_id).scan().to_arrow()) == 3

authn_results

## 8. Prepare OpenFGA Authorization Configuration



The opt-in `docker-compose-authz.yaml` overlay adds OpenFGA and changes only Lakekeeper's authorization settings; all Keycloak settings remain unchanged. Before activation, record the baseline and rollback:



```bash

docker compose ps

docker compose config > /tmp/lakekeeper-authn-only.yaml

docker compose -f docker-compose.yaml -f docker-compose-authz.yaml config > /tmp/lakekeeper-openfga.yaml

diff -u /tmp/lakekeeper-authn-only.yaml /tmp/lakekeeper-openfga.yaml || true

```



Rollback configuration is the base stack: `docker compose up -d --force-recreate lakekeeper`. This restores `allowall`; it does not delete OpenFGA data. Do not use `docker compose down -v` unless deleting all POC state is intentional.

## 9. Enable Authorization



Pause here. From a host terminal in `examples/access-control-simple`, run these commands during a write-free window:



```bash

dc_authz() { docker compose -f docker-compose.yaml -f docker-compose-authz.yaml "$@"; }

docker compose stop lakekeeper

dc_authz up -d openfga

dc_authz run --rm migrate

dc_authz run --rm --entrypoint /home/nonroot/lakekeeper migrate openfga reconcile --mode add-missing

dc_authz run --rm --entrypoint /home/nonroot/lakekeeper migrate reopen-bootstrap --yes

dc_authz up -d lakekeeper

dc_authz ps

```



Migration installs Lakekeeper's authorization model. Reconcile copies catalog hierarchy into OpenFGA, but cannot reconstruct ownership or grants. Reopening bootstrap allows the intended technical principal to seed the new store. Resume below after Lakekeeper is healthy.

In [ ]:
tokens = {name: client_token(*credentials) for name, credentials in CLIENTS.items()}

tokens.update({name: user_token(*credentials) for name, credentials in USERS.items()})

claims = {name: jwt.decode(token, options={"verify_signature": False}) for name, token in tokens.items()}



response = requests.get(f"{MANAGEMENT_URL}/v1/info", headers=auth(tokens["technical"]), timeout=10)

response.raise_for_status()

if not response.json()["bootstrapped"]:

    response = requests.post(

        f"{MANAGEMENT_URL}/v1/bootstrap",

        headers=auth(tokens["technical"]),

        json={"accept-terms-of-use": True},

        timeout=30,

    )

    response.raise_for_status()



technical_user_id = f"oidc~{claims['technical']['sub']}"

peter_user_id = f"oidc~{claims['peter']['sub']}"

anna_user_id = f"oidc~{claims['anna']['sub']}"

trino_user_id = f"oidc~{claims['trino']['sub']}"

print("OpenFGA bootstrap complete for", technical_user_id)

## 10. Verify Authorized and Unauthorized Access



Before grants, Peter, Anna, and the Trino service principal must not read the table. Then the technical administrator receives project administration, Peter receives warehouse `describe` plus table `select`, and Trino receives warehouse `select`. Anna remains ungranted until the final limited-read test.



Because this simple deployment configures one shared Trino catalog credential, Trino tests the `trino` service principal. It cannot distinguish Peter from Anna. End-user authorization is verified directly through PyIceberg; use the advanced example's identity propagation/OPA pattern when a shared query engine must enforce end-user identity.

In [ ]:
def expect_denied(label, operation):

    try:

        operation()

    except Exception as error:

        print(f"DENIED as expected: {label}: {type(error).__name__}")

        return

    raise AssertionError(f"Expected authorization denial: {label}")



expect_denied("Peter PyIceberg read", lambda: catalog_for_token("peter").load_table(table_id))

expect_denied("Anna PyIceberg read", lambda: catalog_for_token("anna").load_table(table_id))



def trino_rows():

    connection = connect(host=TRINO_HOST, port=TRINO_PORT, user="poc", catalog="lakekeeper")

    return connection.cursor().execute("SELECT event_id, source, amount FROM authn_poc.sample_events ORDER BY event_id").fetchall()



expect_denied("Trino service read", trino_rows)

In [ ]:
response = requests.post(

    f"{MANAGEMENT_URL}/v1/permissions/project/assignments",

    headers=auth(tokens["technical"]),

    json={"writes": [{"type": "project_admin", "user": technical_user_id}]},

    timeout=10,

)

response.raise_for_status()



table_uuid = str(table.metadata.table_uuid)

grants = {

    f"{MANAGEMENT_URL}/v1/warehouse/{warehouse_id}/grants": [

        {"privilege": "describe", "principal": {"user": peter_user_id}},

        {"privilege": "select", "principal": {"user": trino_user_id}},

    ],

    f"{MANAGEMENT_URL}/v1/warehouse/{warehouse_id}/table/{table_uuid}/grants": [

        {"privilege": "select", "principal": {"user": peter_user_id}},

    ],

}

for url, writes in grants.items():

    response = requests.post(url, headers=auth(tokens["technical"]), json={"writes": writes}, timeout=10)

    response.raise_for_status()



assert catalog_for_token("peter").load_table(table_id).scan().to_arrow().to_pylist() == expected_rows

expect_denied("Anna remains ungranted", lambda: catalog_for_token("anna").load_table(table_id))

assert trino_rows() == [(1, "pyiceberg", 10.5), (2, "pyiceberg", 20.0), (3, "trino-ready", 31.5)]

print("Peter and Trino are allowed; Anna is denied")

In [ ]:
response = requests.post(

    f"{MANAGEMENT_URL}/v1/warehouse/{warehouse_id}/table/{table_uuid}/grants",

    headers=auth(tokens["technical"]),

    json={"writes": [{"privilege": "select", "principal": {"user": anna_user_id}}]},

    timeout=10,

)

response.raise_for_status()

assert catalog_for_token("anna").load_table(table_id).scan().to_arrow().to_pylist() == expected_rows



response = requests.get(

    f"{MANAGEMENT_URL}/v1/warehouse/{warehouse_id}/table/{table_uuid}/grants",

    headers=auth(tokens["technical"]), timeout=10,

)

response.raise_for_status()

display(JSON(response.json()))

print("Anna's limited table read is now allowed")

## 11. README and Commit Commands



The adjacent README contains startup, baseline verification, cutover, troubleshooting, and rollback instructions. Review and commit from the repository root:



```bash

git status --short

git diff -- examples/access-control-simple

git branch --show-current

git add examples/access-control-simple

git commit -m "docs: add staged authn to authz poc"

```